# Anton calibration: real GPU numbers for the cost model

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU). It measures the three numbers Anton's planner and mock replicas need:

1. **Prefill time per token**: how long the GPU takes to process context that is *not* in the KV cache.
2. **KV cache bytes per token**: how big a session's cached state is.
3. **Offload / restore throughput**: how fast that state moves off the GPU and back.

It uses Hugging Face `transformers` directly, not vLLM, so prefill here is somewhat slower than a tuned vLLM server would be. Say that when you quote the numbers.

The last cell prints a ConfigMap block to paste into `deploy/k8s/anton.yaml`.

In [ ]:
!pip -q install -U transformers accelerate
import torch, time, io, json
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated; swap for a larger model if the GPU has room
assert torch.cuda.is_available(), "Switch the runtime to GPU first"
print(torch.cuda.get_device_name(0))
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda").eval()

In [ ]:
def kv_tensors(pkv):
    """Return every key/value tensor in a KV cache, across transformers versions."""
    out = []
    def walk(x):
        if torch.is_tensor(x): out.append(x)
        elif isinstance(x, (list, tuple)): [walk(i) for i in x]
        elif hasattr(x, "layers"): [walk(vars(l)) for l in x.layers]
        elif hasattr(x, "key_cache"): walk(list(x.key_cache)); walk(list(x.value_cache))
        elif isinstance(x, dict): [walk(v) for v in x.values()]
    walk(pkv)
    return out

@torch.no_grad()
def prefill(n_tokens):
    ids = torch.randint(1000, 20000, (1, n_tokens), device="cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = model(ids, use_cache=True)
    torch.cuda.synchronize()
    return time.perf_counter() - t0, out.past_key_values

prefill(256)  # warm-up
rows = []
for n in [1000, 2000, 4000, 8000, 12000, 16000]:
    try:
        best = min(prefill(n)[0] for _ in range(3))
        secs, pkv = prefill(n)
        tensors = kv_tensors(pkv)
        size = sum(t.numel() * t.element_size() for t in tensors)

        torch.cuda.synchronize(); t0 = time.perf_counter()
        cpu = [t.cpu() for t in tensors]                 # offload: GPU -> host memory
        buf = io.BytesIO(); torch.save(cpu, buf)         # ...and serialize, as a cache tier would receive it
        off = time.perf_counter() - t0

        t0 = time.perf_counter()
        buf.seek(0); back = [t.cuda() for t in torch.load(buf)]   # restore: deserialize -> GPU
        torch.cuda.synchronize(); res = time.perf_counter() - t0

        rows.append(dict(tokens=n, prefill_s=best, kv_bytes=size, offload_s=off, restore_s=res))
        print(f"{n:>6} tokens | prefill {best*1000:8.1f} ms | KV {size/1e6:8.1f} MB | offload {off*1000:7.1f} ms | restore {res*1000:7.1f} ms")
        del pkv, tensors, cpu, back, buf; torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        print(f"{n} tokens: out of GPU memory, stopping"); torch.cuda.empty_cache(); break

In [ ]:
# Use the largest context that fit: that is the regime agent sessions live in.
r = rows[-1]
ms_per_token   = r["prefill_s"] * 1000 / r["tokens"]
bytes_per_tok  = r["kv_bytes"] / r["tokens"]
offload_bps    = r["kv_bytes"] / r["offload_s"]
restore_bps    = r["kv_bytes"] / r["restore_s"]
move_s, recompute_s = r["offload_s"] + r["restore_s"], r["prefill_s"]
print(f"Offload + restore takes {move_s:.2f}s vs {recompute_s:.2f}s to recompute "
      f"({'moving wins' if move_s < recompute_s else 'recompute wins'} at {r['tokens']} tokens)\n")
print("Paste into the anton-sim-params ConfigMap in deploy/k8s/anton.yaml:\n")
print(f'  PREFILL_MS_PER_TOKEN: "{ms_per_token:.4f}"')
print(f'  BYTES_PER_TOKEN: "{bytes_per_tok:.0f}"')
print(f'  OFFLOAD_BYTES_PER_SEC: "{offload_bps:.0f}"')
print(f'  RESTORE_BYTES_PER_SEC: "{restore_bps:.0f}"')
print(f"\nAlso set the router flag --prefill-ms-per-token={ms_per_token:.4f}")
print("\nRaw rows:", json.dumps(rows))